In [ ]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

In [ ]:
import math
import sys
sys.path.insert(0, '/home/t/orbitwars')
from visualizer import Visualizer
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

viz = Visualizer()

def hellburner(obs):
    moves = []
    player = obs["player"]
    planets = [Planet(*p) for p in obs["planets"]]

    viz.record(obs)
    step = obs["step"]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    for mine in my_planets:
        nearest = None
        min_dist = float('inf')
        for t in targets:
            dist = math.sqrt((mine.x - t.x)**2 + (mine.y - t.y)**2)
            if dist < min_dist:
                min_dist = dist
                nearest = t

        if nearest is None:
            continue

        ships_needed = max(nearest.ships + 1, 20)

        if mine.ships >= ships_needed:
            angle = math.atan2(nearest.y - mine.y, nearest.x - mine.x)
            viz.add_line(step, mine.x, mine.y, nearest.x, nearest.y, color="#ff4444", width=1)
            viz.add_text(step, f"P{mine.id} -> P{nearest.id} ({ships_needed} ships, dist={min_dist:.1f})")
            moves.append([mine.id, angle, ships_needed])

    return moves


In [ ]:
# Run game and save visualizer
env = make('orbit_wars', debug=False)
env.run([hellburner, 'random'])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f'Player {i}: reward={s.reward}, status={s.status}')

#env.render(mode='ipython', width=800, height=600)
viz.save('/mnt/c/Users/ajohn/Downloads/orbitwars_viz.html')